# 🕸️ Web Scraping Notebook
Collect customer reviews from Yelp, Trustpilot, TripAdvisor, and Amazon.

## Requirements
```bash
pip install selenium webdriver-manager python-dotenv
```
For Yelp API: add `YELP_API_KEY=your_key` to your `.env` file.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
from scraping.scraper_pipeline import ScraperPipeline

pipeline = ScraperPipeline(output_dir='../data')
print('Pipeline ready ✅')

## Option 1 — Yelp API (Easiest, No Selenium)

In [ ]:
from scraping.yelp_api import YelpScraper

# Requires YELP_API_KEY in .env file
yelp = YelpScraper()

# Search + collect reviews
df_yelp = yelp.collect(
    term='restaurants',
    location='Tunis, Tunisia',
    n_businesses=10,
)

print(df_yelp.head())
df_yelp.to_csv('../data/yelp_reviews.csv', index=False)

## Option 2 — Trustpilot (No Selenium needed)

In [ ]:
from scraping.trustpilot_scraper import TrustpilotScraper

scraper = TrustpilotScraper(delay=2.0)

# Scrape multiple companies
df_tp = scraper.collect_multiple(
    companies=['amazon.com', 'booking.com', 'airbnb.com'],
    max_pages=3,   # ~20 reviews per page
)

print(f'Collected: {len(df_tp)} reviews')
print(df_tp['company'].value_counts())
df_tp.to_csv('../data/trustpilot_reviews.csv', index=False)

## Option 3 — TripAdvisor (Selenium required)

In [ ]:
from scraping.tripadvisor_scraper import TripAdvisorScraper

scraper = TripAdvisorScraper(headless=True, delay=3.0)

# Example: Hotel in Tunis
# Find the URL by searching on tripadvisor.com and copying the hotel URL
urls = [
    "https://www.tripadvisor.com/Hotel_Review-g293758-d301898-Reviews-Hotel_Africa-Tunis_Tunis_Governorate.html",
    # Add more hotel/restaurant URLs here
]

names = ["Hotel Africa Tunis"]

df_ta = scraper.collect_multiple(urls, max_pages=3, names=names)
df_ta.to_csv('../data/tripadvisor_reviews.csv', index=False)

## Option 4 — Amazon (Selenium required)

In [ ]:
from scraping.amazon_scraper import AmazonScraper

scraper = AmazonScraper(headless=True, delay=3.5)

# Find ASIN in any Amazon product URL after /dp/
# Example: https://www.amazon.com/dp/B08N5WRWNW
asins = [
    "B08N5WRWNW",   # Echo Dot 4th Gen
    "B0BQZSWT8N",   # Another product
]
names = ["Echo Dot 4th Gen", "Product 2"]

df_amz = scraper.collect_multiple(asins, max_pages=5, names=names)
df_amz.to_csv('../data/amazon_reviews.csv', index=False)

## Full Pipeline (all sources combined)

In [ ]:
# Run everything in one call
df_all = pipeline.run(
    # Yelp (requires API key in .env)
    yelp_term='restaurants',
    yelp_location='Tunis, Tunisia',
    yelp_n_businesses=10,

    # Trustpilot
    trustpilot_companies=['booking.com', 'amazon.com'],
    trustpilot_max_pages=3,

    # TripAdvisor (optional — needs Selenium + URL)
    # tripadvisor_urls=["https://..."],

    # Amazon (optional — needs Selenium + ASIN)
    # amazon_asins=["B08N5WRWNW"],

    save_csv=True,
    filename='all_reviews.csv',
)

print(df_all.info())
df_all.head()

## Quick Stats on Collected Data

In [ ]:
import matplotlib.pyplot as plt

print(f'Total reviews: {len(df_all)}')
print('\nBy platform:')
print(df_all['platform'].value_counts())
print('\nRating distribution:')
print(df_all['rating'].value_counts().sort_index())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df_all['platform'].value_counts().plot(kind='bar', ax=axes[0], color='#6366f1')
axes[0].set_title('Reviews by Platform')
axes[0].tick_params(rotation=45)

df_all['rating'].value_counts().sort_index().plot(kind='bar', ax=axes[1], color='#06b6d4')
axes[1].set_title('Rating Distribution')
plt.tight_layout()
plt.savefig('../assets/scraping_stats.png', dpi=150)
plt.show()